In [1]:
import pandas as pd
import numpy as np
import re

# Load dataset (adjust path depending on notebook location)
df = pd.read_csv('../data/attacks.csv', encoding='latin1')

# Strip trailing and leading whitespace from column names
df.columns = df.columns.str.strip()

# Filter for post-1900 data
df = df[df['Year'] >= 1900]

# Check shape and column names
print(f"Data shape: {df.shape}")
df.head()

Data shape: (5409, 22)


,Case Number,Date,Year,Type,Country,Area,Location,Activity,Name,Sex,...,Fatal (Y/N),Time,Species,Investigator or Source,pdf,href formula,href,Case Number.1,Case Number.2,original order
0,2017.06.11,2017-06-11,2017.0,Unprovoked,AUSTRALIA,Western Australia,"Point Casuarina, Bunbury",Body boarding,Paul Goff,M,...,N,08h30,"White shark, 4 m","WA Today, 6/11/2017",2017.06.11-Goff.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.11,2017.06.11,6095.0
1,2017.06.10.b,2017-06-10,2017.0,Unprovoked,AUSTRALIA,Victoria,"Flinders, Mornington Penisula",Surfing,female,F,...,N,15h45,7 gill shark,NaN,2017.06.10.b-Flinders.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.10.b,2017.06.10.b,6094.0
2,2017.06.10.a,2017-06-10,2017.0,Unprovoked,USA,Florida,"Ponce Inlet, Volusia County",Surfing,Bryan Brock,M,...,N,10h00,NaN,"Daytona Beach News-Journal, 6/10/2017",2017.06.10.a-Brock.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.10.a,2017.06.10.a,6093.0
3,2017.06.07.R,Reported 07-Jun-2017,2017.0,Unprovoked,UNITED KINGDOM,South Devon,Bantham Beach,Surfing,Rich Thomson,M,...,N,NaN,"3m shark, probably a smooth hound","C. Moore, GSAF",2017.06.07.R-Thomson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.07.R,2017.06.07.R,6092.0
4,2017.06.04,2017-06-04,2017.0,Unprovoked,USA,Florida,"Middle Sambo Reef off Boca Chica, Monroe County",Spearfishing,Parker Simpson,M,...,N,NaN,8' shark,"Nine News, 6/7/2017",2017.06.04-Simpson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2017.06.04,2017.06.04,6091.0


In [2]:
# Check all column names
print("Columns in dataset:")
print(df.columns.tolist())

# Inspect non-null values and data types
df.info()


Columns in dataset:
['Case Number', 'Date', 'Year', 'Type', 'Country', 'Area', 'Location', 'Activity', 'Name', 'Sex', 'Age', 'Injury', 'Fatal (Y/N)', 'Time', 'Species', 'Investigator or Source', 'pdf', 'href formula', 'href', 'Case Number.1', 'Case Number.2', 'original order']
<class 'pandas.core.frame.DataFrame'>
Index: 5409 entries, 0 to 5410
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Case Number             5409 non-null   object 
 1   Date                    5409 non-null   object 
 2   Year                    5409 non-null   float64
 3   Type                    5407 non-null   object 
 4   Country                 5387 non-null   object 
 5   Area                    5113 non-null   object 
 6   Location                5041 non-null   object 
 7   Activity                4979 non-null   object 
 8   Name                    5246 non-null   object 
 9   Sex                     4898 non

In [3]:
# 1. Clean Time Column into standardized buckets
def standardize_time(val):
    val = str(val).lower()
    match = re.search(r'(\d{2})[h:]?(\d{2})?', val)
    if match:
        hour = int(match.group(1))
        if 0 <= hour <= 23:
            if 5 <= hour < 12:
                return 'Morning (05:00-11:59)'
            elif 12 <= hour < 17:
                return 'Afternoon (12:00-16:59)'
            elif 17 <= hour < 21:
                return 'Evening (17:00-20:59)'
            else:
                return 'Night (21:00-04:59)'
    
    if 'morning' in val or 'dawn' in val:
        return 'Morning (05:00-11:59)'
    elif 'afternoon' in val or 'noon' in val or 'lunch' in val:
        return 'Afternoon (12:00-16:59)'
    elif 'evening' in val or 'dusk' in val or 'sunset' in val:
        return 'Evening (17:00-20:59)'
    elif 'night' in val or 'dark' in val:
        return 'Night (21:00-04:59)'
    else:
        return 'Unknown'

df['Time_Category'] = df['Time'].apply(standardize_time)

# 2. Extract primary shark species
def extract_species(val):
    val = str(val).lower()
    if 'white' in val:
        return 'White Shark'
    elif 'tiger' in val:
        return 'Tiger Shark'
    elif 'bull' in val:
        return 'Bull Shark'
    elif 'blacktip' in val:
        return 'Blacktip Shark'
    elif 'nurse' in val:
        return 'Nurse Shark'
    elif 'wobbegong' in val:
        return 'Wobbegong Shark'
    elif 'hammerhead' in val:
        return 'Hammerhead Shark'
    elif 'mako' in val:
        return 'Mako Shark'
    else:
        return 'Unidentified / Other'

df['Clean_Species'] = df['Species'].apply(extract_species)

# Verify results
df[['Time', 'Time_Category', 'Species', 'Clean_Species']].head(10)

,Time,Time_Category,Species,Clean_Species
0,08h30,Morning (05:00-11:59),"White shark, 4 m",White Shark
1,15h45,Afternoon (12:00-16:59),7 gill shark,Unidentified / Other
2,10h00,Morning (05:00-11:59),NaN,Unidentified / Other
3,NaN,Unknown,"3m shark, probably a smooth hound",Unidentified / Other
4,NaN,Unknown,8' shark,Unidentified / Other
5,Shortly before 12h00,Afternoon (12:00-16:59),Tiger shark,Tiger Shark
6,NaN,Unknown,3' shark,Unidentified / Other
7,Morning,Morning (05:00-11:59),Tiger shark,Tiger Shark
8,NaN,Unknown,NaN,Unidentified / Other
9,Morning,Morning (05:00-11:59),NaN,Unidentified / Other


In [4]:
# 1. Standardize Geographic Data
df['Country'] = df['Country'].astype(str).str.strip().str.upper()
df['Area'] = df['Area'].astype(str).str.strip().str.title()
df['Location'] = df['Location'].astype(str).str.strip().str.title()

# 2. Extract Body Part Injured from Injury Column
def extract_body_part(val):
    val = str(val).lower()
    parts = []
    if any(k in val for k in ['leg', 'thigh', 'knee', 'calf', 'shin', 'foot', 'ankle', 'toe']):
        parts.append('Leg / Foot')
    if any(k in val for k in ['arm', 'hand', 'finger', 'wrist', 'elbow', 'shoulder']):
        parts.append('Arm / Hand')
    if any(k in val for k in ['torso', 'chest', 'back', 'abdomen', 'stomach', 'rib']):
        parts.append('Torso')
    if any(k in val for k in ['head', 'neck', 'face']):
        parts.append('Head / Neck')
    
    if not parts:
        return 'Other / Unspecified'
    return ' & '.join(parts)

df['Body_Part_Injured'] = df['Injury'].apply(extract_body_part)

# 3. Clean Fatality Indicator
def clean_fatal(val):
    val = str(val).strip().upper()
    if val in ['Y', 'FATAL']:
        return 'Y'
    elif val in ['N', '2017']: # '2017' is a known typo in raw GSAF data
        return 'N'
    else:
        return 'UNKNOWN'

df['Clean_Fatal'] = df['Fatal (Y/N)'].apply(clean_fatal)

# Preview results
df[['Country', 'Area', 'Injury', 'Body_Part_Injured', 'Clean_Fatal']].head()

,Country,Area,Injury,Body_Part_Injured,Clean_Fatal
0,AUSTRALIA,Western Australia,"No injury, board bitten",Other / Unspecified,N
1,AUSTRALIA,Victoria,"No injury, knocke off board",Other / Unspecified,N
2,USA,Florida,Laceration to left foot,Leg / Foot,N
3,UNITED KINGDOM,South Devon,"Bruise to leg, cuts to hand sustained when he ...",Leg / Foot & Arm / Hand,N
4,USA,Florida,Laceration to shin,Leg / Foot,N


In [5]:
# Save cleaned dataframe for SQL and Power BI
output_path = '../data/clean_attacks.csv'
df.to_csv(output_path, index=False)

print(f"Data cleaning complete! Cleaned dataset saved to {output_path}")
print(f"Total cleaned records post-1900: {len(df)}")

Data cleaning complete! Cleaned dataset saved to ../data/clean_attacks.csv
Total cleaned records post-1900: 5409


In [7]:
import sqlite3

# 1. Create an in-memory SQL database connection
conn = sqlite3.connect(':memory:')

# 2. Push your cleaned dataframe into a SQL table named 'shark_attacks'
df.to_sql('shark_attacks', conn, index=False, if_exists='replace')

# 3. SQL Query: Annual trend of shark attacks since 1900
sql_query_annual = """
SELECT 
    Year, 
    COUNT(*) AS Total_Attacks
FROM shark_attacks
GROUP BY Year
ORDER BY Year ASC;
"""

annual_trends = pd.read_sql_query(sql_query_annual, conn)
annual_trends.head(10)

,Year,Total_Attacks
0,1900.0,13
1,1901.0,9
2,1902.0,15
3,1903.0,10
4,1904.0,13
5,1905.0,16
6,1906.0,21
7,1907.0,23
8,1908.0,13
9,1909.0,18


In [9]:
# Query 1: Annual Trends Since 1900
sql_query_annual = """
SELECT 
    Year, 
    COUNT(*) AS Total_Attacks
FROM shark_attacks
GROUP BY Year
ORDER BY Year ASC;
"""

annual_trends = pd.read_sql_query(sql_query_annual, conn)
annual_trends.head(10)

,Year,Total_Attacks
0,1900.0,13
1,1901.0,9
2,1902.0,15
3,1903.0,10
4,1904.0,13
5,1905.0,16
6,1906.0,21
7,1907.0,23
8,1908.0,13
9,1909.0,18


In [10]:
# Query 2: Which countries report the most shark attacks?
sql_query_countries = """
SELECT 
    Country, 
    COUNT(*) AS Total_Attacks
FROM shark_attacks
WHERE Country IS NOT NULL AND Country != ''
GROUP BY Country
ORDER BY Total_Attacks DESC
LIMIT 10;
"""

top_countries = pd.read_sql_query(sql_query_countries, conn)
top_countries

,Country,Total_Attacks
0,USA,2030
1,AUSTRALIA,1147
2,SOUTH AFRICA,541
3,PAPUA NEW GUINEA,129
4,BRAZIL,101
5,NEW ZEALAND,95
6,BAHAMAS,93
7,MEXICO,77
8,ITALY,61
9,REUNION,57


In [11]:
# Query 3: Within those top countries, which areas/locations are the most dangerous?
sql_query_dangerous_areas = """
SELECT 
    Country, 
    Area, 
    COUNT(*) AS Total_Attacks
FROM shark_attacks
WHERE Country IN ('USA', 'AUSTRALIA', 'SOUTH AFRICA')
  AND Area IS NOT NULL AND Area != ''
GROUP BY Country, Area
ORDER BY Total_Attacks DESC
LIMIT 15;
"""

dangerous_areas = pd.read_sql_query(sql_query_dangerous_areas, conn)
dangerous_areas

,Country,Area,Total_Attacks
0,USA,Florida,988
1,AUSTRALIA,New South Wales,413
2,AUSTRALIA,Queensland,277
3,USA,California,275
4,USA,Hawaii,267
5,SOUTH AFRICA,Kwazulu-Natal,197
6,SOUTH AFRICA,Western Cape Province,183
7,AUSTRALIA,Western Australia,165
8,SOUTH AFRICA,Eastern Cape Province,150
9,USA,South Carolina,133


In [12]:
# Query 4: What body parts are most often injured?
sql_query_injuries = """
SELECT 
    Body_Part_Injured, 
    COUNT(*) AS Total_Incidents
FROM shark_attacks
WHERE Body_Part_Injured IS NOT NULL
GROUP BY Body_Part_Injured
ORDER BY Total_Incidents DESC;
"""

injured_body_parts = pd.read_sql_query(sql_query_injuries, conn)
injured_body_parts

,Body_Part_Injured,Total_Incidents
0,Other / Unspecified,2114
1,Leg / Foot,2010
2,Arm / Hand,785
3,Leg / Foot & Arm / Hand,193
4,Torso,107
5,Leg / Foot & Torso,49
6,Head / Neck,48
7,Arm / Hand & Torso,45
8,Leg / Foot & Arm / Hand & Torso,21
9,Arm / Hand & Head / Neck,16


In [13]:
# Query 5: Are attacks more common during certain parts of the day?
sql_query_time = """
SELECT 
    Time_Category, 
    COUNT(*) AS Total_Attacks
FROM shark_attacks
WHERE Time_Category != 'Unknown'
GROUP BY Time_Category
ORDER BY Total_Attacks DESC;
"""

time_analysis = pd.read_sql_query(sql_query_time, conn)
time_analysis

,Time_Category,Total_Attacks
0,Afternoon (12:00-16:59),1293
1,Morning (05:00-11:59),865
2,Evening (17:00-20:59),439
3,Night (21:00-04:59),105


In [15]:
# Query 6: Which species of shark are attacking most often?
sql_query_species = """
SELECT 
    Clean_Species, 
    COUNT(*) AS Total_Attacks
FROM shark_attacks
WHERE Clean_Species != 'Unidentified / Other'
GROUP BY Clean_Species
ORDER BY Total_Attacks DESC;
"""

top_species = pd.read_sql_query(sql_query_species, conn)
top_species

,Clean_Species,Total_Attacks
0,White Shark,643
1,Tiger Shark,280
2,Bull Shark,165
3,Blacktip Shark,101
4,Nurse Shark,91
5,Mako Shark,54
6,Wobbegong Shark,45
7,Hammerhead Shark,44
